# Data Bit I: Metrics computation

In [1]:

import pandas as pd
import numpy as np
 
df = pd.read_csv("../data/affected_km2_per_day.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

In [ ]:
#growth: 
first_week_avg = df.head(7)["affected_km2"].mean()
last_week_avg  = df.tail(7)["affected_km2"].mean()
overall_growth = (last_week_avg - first_week_avg) / first_week_avg * 100
print(f"\nFirst week avg: {first_week_avg:,.0f} km²")
print(f"Last week avg: {last_week_avg:,.0f} km²")
print(f"Overall change: {overall_growth:+.1f}%")



First week avg:  518,206 km²
Last week avg:   1,871,510 km²
Overall change:  +261.2%


In [ ]:
#peak
peak_row = df.loc[df["affected_km2"].idxmax()]
print(f"\nAll-time peak: {peak_row['affected_km2']:,.0f} km² on {peak_row['date'].date()}")
print(f"{peak_row['affected_km2'] / 510_072_000 * 100:.2f}% of Earth's surface")


All-time peak: 2,204,083 km²  on 2025-06-18
0.43% of Earth's surface


In [4]:
#Annual avg
annual = (
    df.groupby(df["date"].dt.year)["affected_km2"]
    .mean()
    .round(0)
    .rename("avg_affected_km2")
)
print(f"\nAnnual averages:")
print(annual.to_string())


Annual averages:
date
2022     504157.0
2023     919863.0
2024    1468787.0
2025    1652688.0
2026    1697365.0


In [5]:
#conflict event windows
events = {
    "Ukraine invasion (Feb–Apr 2022)":   ("2022-02-24", "2022-04-30"),
    "Israel–Gaza conflict (Oct 2023+)":  ("2023-10-07", "2024-01-31"),
}
 
print(f"\nEvent window averages:")
baseline = df[df["date"] < "2022-02-24"]["affected_km2"].mean()
print(f"  Pre-war baseline (before 2022-02-24): {baseline:,.0f} km²")
for label, (start, end) in events.items():
    window = df[(df["date"] >= start) & (df["date"] <= end)]
    avg    = window["affected_km2"].mean()
    vs_baseline = (avg - baseline) / baseline * 100
    print(f"  {label}: {avg:,.0f} km²  ({vs_baseline:+.1f}% vs baseline)")
 
 


Event window averages:
  Pre-war baseline (before 2022-02-24): 529,334 km²
  Ukraine invasion (Feb–Apr 2022): 451,412 km²  (-14.7% vs baseline)
  Israel–Gaza conflict (Oct 2023+): 1,073,000 km²  (+102.7% vs baseline)
